# 03 Evaluate Results

Evaluation runner. It reads the latest dataset and training configs, reconstructs the system, loads checkpoints, evaluates derivative MSE, rollout error, and Lyapunov diagnostics, then saves a JSON summary.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / "stable_icnn_physics").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

for module_name in list(sys.modules):
    if module_name == "stable_icnn_physics" or module_name.startswith("stable_icnn_physics."):
        del sys.modules[module_name]

import json
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyTorch is not installed in this notebook kernel. Install dependencies or switch kernels.") from exc

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model, make_system
from stable_icnn_physics.data import load_dataset, load_trajectory_dataset, tensor_dataset
from stable_icnn_physics.eval import autoregressive_rollout_model, lyapunov_decrease_values, rollout_error, rollout_system
from stable_icnn_physics.plotting import plot_lyapunov_contours, plot_vector_field
from stable_icnn_physics.train import evaluate_derivative_mse

CACHE_DIR = REPO_ROOT / "data" / "cache"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"
DATASET_CONFIG_PATH = CACHE_DIR / "latest_dataset_config.json"
TRAINING_CONFIG_PATH = CHECKPOINT_DIR / "latest_training_config.json"

ROLLOUT_STEPS = None  # None means use min(dataset steps, 300) for trajectory datasets, else 300
N_ROLLOUTS = 16
TOLERANCE = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not DATASET_CONFIG_PATH.exists() or not TRAINING_CONFIG_PATH.exists():
    raise FileNotFoundError("Run 01_generate_data.ipynb and 02_train_models.ipynb before evaluation.")
dataset_config = json.loads(DATASET_CONFIG_PATH.read_text(encoding="utf-8"))
training_config = json.loads(TRAINING_CONFIG_PATH.read_text(encoding="utf-8"))

system_meta = dataset_config["system"]
if system_meta.get("factory_name") == "custom_state_space" or system_meta.get("rhs_serialized") is False:
    raise ValueError("Custom callable systems cannot be reconstructed from JSON alone. Define the callable here and call make_system manually.")
system = make_system(system_meta["name"], **system_meta.get("params", {}))
state_dim = int(system_meta["state_dim"])
hparams = training_config["hyperparameters"]
DT = float(dataset_config.get("dt") or 0.02)
max_steps = int(dataset_config.get("steps") or 300)
ROLLOUT_STEPS = min(max_steps, 300) if ROLLOUT_STEPS is None else int(ROLLOUT_STEPS)
print("system:", system.name, "state_dim:", state_dim, "dt:", DT, "rollout steps:", ROLLOUT_STEPS, "device:", DEVICE)


## Load Test Data And Models

In [ ]:
test_dataset_path = Path(dataset_config["test_dataset_path"])
x_test, y_test = load_dataset(test_dataset_path)
test_ds = tensor_dataset(x_test, y_test)

stable_model = build_stable_model(
    dim=state_dim,
    hidden=hparams["hidden"],
    depth=hparams["depth"],
    lyapunov_hidden=hparams["lyapunov_hidden"],
    lyapunov_eps=hparams["lyapunov_eps"],
    alpha=hparams["alpha"],
    rehu_width=hparams.get("rehu_width", 0.01),
)
baseline_model = BaselineDynamicsMLP(dim=state_dim, hidden=hparams["hidden"], depth=hparams["depth"])

stable_ckpt_path = Path(training_config["stable_checkpoint_path"])
baseline_ckpt_path = Path(training_config["baseline_checkpoint_path"])
stable_model.load_state_dict(torch.load(stable_ckpt_path, map_location=DEVICE)["model_state"])
baseline_model.load_state_dict(torch.load(baseline_ckpt_path, map_location=DEVICE)["model_state"])
stable_model.to(DEVICE).eval()
baseline_model.to(DEVICE).eval()

print("test:", x_test.shape, y_test.shape)
print("stable checkpoint:  ", stable_ckpt_path)
print("baseline checkpoint:", baseline_ckpt_path)


## Derivative MSE

In [ ]:
derivative_mse_stable = evaluate_derivative_mse(stable_model, test_ds, device=DEVICE)
derivative_mse_baseline = evaluate_derivative_mse(baseline_model, test_ds, device=DEVICE)
print("stable derivative MSE:  ", derivative_mse_stable)
print("baseline derivative MSE:", derivative_mse_baseline)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["stable", "baseline"], [derivative_mse_stable, derivative_mse_baseline], color=["tab:blue", "tab:orange"])
ax.set_yscale("log")
ax.set_ylabel("test derivative MSE")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()


## Rollout Evaluation

In [ ]:
trajectory_data = None
if dataset_config.get("test_trajectory_path"):
    trajectory_path = Path(dataset_config["test_trajectory_path"])
    if trajectory_path.exists():
        trajectory_data = load_trajectory_dataset(trajectory_path)

if trajectory_data is not None and "trajectories" in trajectory_data:
    saved = trajectory_data["trajectories"][:N_ROLLOUTS, : ROLLOUT_STEPS + 1, :]
    x0 = saved[:, 0, :]
    true_traj = np.transpose(saved, (1, 0, 2)).astype(np.float32)
else:
    x0 = system.sample_initial_conditions(N_ROLLOUTS, split="test", seed=int(dataset_config.get("seed", 0)) + 123)
    true_traj = rollout_system(system, x0, steps=ROLLOUT_STEPS, dt=DT)

stable_traj = autoregressive_rollout_model(stable_model, x0, steps=ROLLOUT_STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state)
baseline_traj = autoregressive_rollout_model(baseline_model, x0, steps=ROLLOUT_STEPS, dt=DT, device=DEVICE, wrap_fn=system.wrap_state)

stable_error = rollout_error(system, true_traj, stable_traj)
baseline_error = rollout_error(system, true_traj, baseline_traj)
stable_mean_error = stable_error.mean(axis=1)
baseline_mean_error = baseline_error.mean(axis=1)
time = np.arange(ROLLOUT_STEPS + 1) * DT

print("final stable mean error:  ", stable_mean_error[-1])
print("final baseline mean error:", baseline_mean_error[-1])


## Rollout Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time, stable_mean_error, label="stable")
axes[0].plot(time, baseline_mean_error, label="baseline")
axes[0].set_yscale("log")
axes[0].set_xlabel("time")
axes[0].set_ylabel("mean state squared error")
axes[0].set_title("Rollout error")
axes[0].legend()
axes[0].grid(alpha=0.25)

traj_id = 0
coord_count = min(state_dim, 4)
for j in range(coord_count):
    axes[1].plot(time, true_traj[:, traj_id, j], color="black", lw=2 if j == 0 else 1, alpha=0.75, label="true" if j == 0 else None)
    axes[1].plot(time, stable_traj[:, traj_id, j], color="tab:blue", alpha=0.8, label="stable" if j == 0 else None)
    axes[1].plot(time, baseline_traj[:, traj_id, j], color="tab:orange", alpha=0.8, label="baseline" if j == 0 else None)
axes[1].set_xlabel("time")
axes[1].set_title(f"Representative trajectory, first {coord_count} coordinates")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()


## 2D Vector Fields And Lyapunov Diagnostics

In [ ]:
if state_dim == 2:
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    plot_vector_field(system, model=None, ax=axes[0], title="True dynamics")
    plot_vector_field(system, model=stable_model, ax=axes[1], title="Stable learned dynamics")
    plot_vector_field(system, model=baseline_model, ax=axes[2], title="Baseline learned dynamics")
    plot_lyapunov_contours(stable_model, ax=axes[3], title="Learned Lyapunov V")
    plt.tight_layout()
else:
    print("Skipping vector field and contour plots for state_dim != 2.")

decrease = lyapunov_decrease_values(stable_model, x_test[: min(2048, len(x_test))], device=DEVICE).ravel()
lyapunov_max_violation = float(decrease.max())
lyapunov_fraction_satisfied = float(np.mean(decrease <= TOLERANCE))
print("max gradV*f + alpha*V:", lyapunov_max_violation)
print("fraction <= tolerance:", lyapunov_fraction_satisfied)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(decrease, bins=60)
ax.axvline(0.0, color="red", lw=1, ls="--")
ax.set_xlabel("gradV*f + alpha*V")
ax.set_title("Lyapunov decrease residual")
ax.grid(alpha=0.25)
plt.tight_layout()


## Save Summary

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "system_name": system.name,
    "dataset_mode": dataset_config["dataset_mode"],
    "derivative_mse_baseline": float(derivative_mse_baseline),
    "derivative_mse_stable": float(derivative_mse_stable),
    "final_rollout_error_baseline": float(baseline_mean_error[-1]),
    "final_rollout_error_stable": float(stable_mean_error[-1]),
    "mean_rollout_error_baseline": float(baseline_mean_error.mean()),
    "mean_rollout_error_stable": float(stable_mean_error.mean()),
    "lyapunov_max_violation": lyapunov_max_violation,
    "lyapunov_fraction_satisfied": lyapunov_fraction_satisfied,
    "rollout_steps": ROLLOUT_STEPS,
    "dt": DT,
    "n_rollouts": int(x0.shape[0]),
}
summary_path = RESULTS_DIR / f"{system.name}_evaluation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
for key, value in summary.items():
    print(f"{key}: {value}")
print("summary:", summary_path)
